# 02 - Run a VLM from Python (`pyneat.genai.VisionLanguageModel`)

A VLM answers questions about an **image**. This notebook loads a VLM, asks about an image directly,
then shows the efficient pattern - **encode the image once, ask many questions** - so you do not
re-encode the same frame for every follow-up.

> Loading a VLM takes a few minutes and several GB of memory (language weights plus a vision
> encoder). Run these cells on the DevKit.

Adapted from `/workspace/core/tutorials/020_run_a_vlm/run_a_vlm.py`, verified against `module.cpp`
and `/workspace/core/include/genai/VisionLanguageModel.h`.

## Direct model vs server, and which handle

Same rule as the LLM notebook: use the **direct** `VisionLanguageModel` for in-process app logic;
use `GenAIServer` only when the boundary is HTTP. We use `VisionLanguageModel` directly here (you
could also use `GenAIModel`, which auto-detects and would report `accepts_image() == True` for a
VLM directory).

## Image format - a correctness detail

VLM image inputs must be **`uint8` HWC RGB** tensors. OpenCV reads images as **BGR**, so convert
before handing the array to the request. The Neat/OpenCV convention: a 3-channel `cv::Mat` is treated
as BGR and converted to RGB internally, but when you pass a **NumPy** array you convert it yourself
with `cv2.cvtColor(img, cv2.COLOR_BGR2RGB)`. Getting this wrong swaps the red/blue channels and
quietly degrades answers.

Source: `GenAITypes.h` ("VLM requests require uint8 HWC RGB tensors") and the 020 tutorial's
`load_rgb_image`.

## Memory implications

- A VLM holds **two** things resident: the language model weights **and** a vision encoder.
- Encoding an image produces image embeddings. With `encode(...)` those embeddings are **cached in
  the model** so repeated questions about the same image skip re-encoding - cheaper and faster.
- `Qwen3-VL-4B-Instruct-GPTQ-a16w4` is the happy-path VLM. If disk is tight (check `df -h /` first),
  `LFM2-VL-450M-a16w4` is the smallest viable VLM.

## Step 1 - Load VLM and image

In [ ]:
import cv2
import numpy as np
import pyneat as neat

MODEL_DIR = "/media/nvme/llima/models/Qwen3-VL-4B-Instruct-GPTQ-a16w4"
IMAGE_PATH = "/workspace/demo-neat/tutorial/assets/images/image.png"  # any local image

def load_rgb(path):
    bgr = cv2.imread(path, cv2.IMREAD_COLOR)
    if bgr is None:
        raise RuntimeError(f"failed to read image: {path}")
    return np.asarray(cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB))   # HWC RGB uint8

model = neat.genai.VisionLanguageModel(MODEL_DIR)
image = load_rgb(IMAGE_PATH)
print("model_id     :", model.model_id())
print("accepts_image:", model.accepts_image())
print("image shape  :", image.shape, image.dtype)   # (H, W, 3) uint8

**Interpretation.** `accepts_image()` must be `True` for a VLM directory. The image is a
`(H, W, 3)` `uint8` array in RGB order - exactly what the request expects.

## Step 2 - Ask with a direct image

The simplest path: attach the image to `request.images` and run once. Best for a one-shot visual
question.

In [ ]:
direct = neat.genai.GenerationRequest()
direct.prompt = "Describe this image in one sentence."
direct.images = [image]          # list of uint8 HWC RGB arrays
direct.max_new_tokens = 96

print("direct image:", model.run(direct).text)

**Interpretation.** `request.images` accepts a list, so you can pass more than one image. Under
the hood the array is converted to a Neat image tensor (BGR->RGB handled for cv::Mat; we already made
it RGB). One `run()` = one encode + one generate.

## Step 3 - Encode once, ask many times

If you will ask several questions about the **same** image, `encode(...)` it once. Then set
`use_cached_images = True` on each follow-up so it reuses the cached embeddings instead of
re-encoding.

In [ ]:
if not model.encode([image]):
    raise RuntimeError("VLM did not accept the image for caching")
print("cached_images =", model.cached_image_count())

q1 = neat.genai.GenerationRequest()
q1.prompt = "What details should I inspect more closely?"
q1.use_cached_images = True
q1.max_new_tokens = 96
print("cached #1:", model.run(q1).text)

q2 = neat.genai.GenerationRequest()
q2.prompt = "Summarize the image in three keywords."
q2.use_cached_images = True
q2.max_new_tokens = 48
print("cached #2:", model.run(q2).text)

**Interpretation.** `encode()` returns `True` when the image was accepted and cached;
`cached_image_count()` confirms how many images are cached. Both cached questions skip re-encoding.
Caveat from the tutorial: some model families do not support cached reuse - if `encode()` returns
`False`, fall back to the direct-image path (Step 2) on every request. Another `encode()` call
**replaces** the cached image.

## Step 4 - Image on a chat message

In a multi-turn conversation, attach the image to the specific `ChatMessage` that needs it, via
`message.images`. This keeps the image next to the exact text it belongs to.

In [ ]:
image_message = neat.genai.ChatMessage()
image_message.role = "user"
image_message.content = "What is the main subject of this image?"
image_message.images = [image]

req = neat.genai.GenerationRequest()
req.messages = [image_message]
req.max_new_tokens = 96
print("message image:", model.run(req).text)

**When to use which image path**

| Situation | Use |
| --- | --- |
| One question about one image | `request.images` (Step 2) |
| Many questions about the **same** image | `encode()` + `use_cached_images=True` (Step 3) |
| A conversation where one turn carries the image | `ChatMessage.images` (Step 4) |
| Each request uses a **different** image | direct `request.images` each time (caching gains nothing) |

## Run it as a script

On the DevKit:

```bash
dk /workspace/demo-neat/llima/02-run-llm-vlm/scripts/run_vlm.py \
  --model /media/nvme/llima/models/Qwen3-VL-4B-Instruct-GPTQ-a16w4 \
  --image /workspace/demo-neat/tutorial/assets/images/image.png \
  --prompt "Describe this image in one sentence."
```

## Expected output

Running on the DevKit produces one answer from a direct image request, multiple follow-up answers
that reuse the cached image, and one answer from a message-level image request:

- **Load:** `accepts_image: True`, image shape `(H, W, 3) uint8`.
- **Direct image:** a one-sentence description of the picture.
- **Cache:** `cached_images = 1`, then two follow-up answers (a "what to inspect" answer and a
  three-keyword summary) that reuse the cached image.
- **Message image:** a short answer naming the main subject.

Exact wording depends on the image and the model's sampling; the shape of the output is what to
verify.

## References

- `/workspace/core/tutorials/020_run_a_vlm/run_a_vlm.py` and `README.md`
- `/workspace/core/include/genai/VisionLanguageModel.h`, `GenAITypes.h` (uint8 HWC RGB requirement)
- `/workspace/core/python/src/module.cpp` (`encode`, `cached_image_count`, `images` bindings)
- `/workspace/core/docs/develop-apps/development-workflow/genai-model.mdx`